# Random Forest Regression Project  
# Medical Cost Prediction & Feature Impact Analysis

---

## **Introduction**

This project focuses on building a deep understanding of how machine learning models can be used to model real world problems, specifically in the healthcare domain. Using the Medical Cost Personal Dataset, the goal is not only to predict insurance charges but also to analyze how different features influence model behavior and performance.

Rather than treating this as a simple regression task, this project is approached as a full machine learning experiment, where different modeling choices, preprocessing techniques, and assumptions are carefully tested and evaluated.

---

## **Project Aim**

The aim of this project is to develop, analyze, and compare multiple regression models to accurately predict medical insurance costs while gaining a strong conceptual and practical understanding of machine learning techniques.

---

## **Objectives**

1. Build a solid understanding of regression algorithms through hands-on implementation
2. Perform in-depth exploratory data analysis (EDA) to uncover patterns and relationships
3. Analyze the effect of key features (especially BMI and smoking) on medical costs
4. Apply and compare different preprocessing techniques:
    - Encoding categorical variables
    - Feature scaling
    - Log transformation (for skewed data)
5. Train and evaluate multiple models, such as:
    - Linear Regression
    - Polynomial Regression
    - Support Vector Regression (SVR)
    - Decision Tree
    - Random Forest
6. Perform model comparison and critical analysis, not just reporting scores
7. Understand why a model performs well or fails, not just the final result

---
## **Dataset Description**

The dataset includes the following features:

- age → Age of the individual
- sex → Gender
- bmi → Body Mass Index
- children → Number of dependents
- smoker → Smoking status
- region → Residential area
- charges → Medical insurance cost (target variable)

The dataset used in this project is available on Kaggle: [Dataset Link](https://www.kaggle.com/datasets/d3lhomi10/medical-cost-personal-dataset)

---

## **Problem Type**

This is a Supervised Machine Learning – Regression Problem, where the goal is to predict a continuous target variable (charges).

---

Project Approach

This project is structured as a complete ML pipeline, including:

1. Data Exploration (EDA)
2. Data Preprocessing
3. Feature Engineering
4. Model Training
5. Model Evaluation
6. Model Comparison
7. Insights & Conclusions

The focus is not only on building a working model, but on understanding each step deeply and justifying every decision.




In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# **Data preprocessing**

## **Data Cleaning Check Template**
This template is designed to quickly assess the quality of any dataset before building machine learning models or performing analysis.

It provides a structured overview of the dataset by checking for common data issues such as:

- Missing values

- Duplicate rows

- Incorrect data types

- Outliers

- Distribution of numerical features

- Categorical feature consistency

**What This Template Does**

- Displays basic dataset information (shape, data types)

- Identifies missing values and duplicates

- Summarizes numerical and categorical features

- Detects potential outliers using the IQR method

- Highlights columns with low unique values for quick inspection

How to Use

1. Load your dataset using Pandas  

2. Call the function:

In [2]:
def data_quality_report(df):

    print("DATA QUALITY REPORT")
    
    # Print a separator line for better readability
    
    print("=" * 50)
    print("BASIC INFO")
    print("=" * 50)
    
    # Show general information about the dataset (columns, data types, non-null values)
    print(df.info())
    
    # Show number of rows and columns
    print("\n" + "=" * 50)
    print("SHAPE OF DATA")
    print("=" * 50)
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    
    # Check for missing (null) values in each column
    print("\n" + "=" * 50)
    print("MISSING VALUES")
    print("=" * 50)
    missing = df.isnull().sum()
    
    # Display only columns that have missing values
    print(missing[missing > 0])
    
    # Check for duplicate rows
    print("\n" + "=" * 50)
    print("DUPLICATES")
    print("=" * 50)
    print(f"Duplicate rows: {df.duplicated().sum()}")
    
    # Display data types of each column
    print("\n" + "=" * 50)
    print("DATA TYPES")
    print("=" * 50)
    print(df.dtypes)
    
    # Summary statistics for numerical columns (mean, std, min, max, etc.)
    print("\n" + "=" * 50)
    print("NUMERICAL SUMMARY")
    print("=" * 50)
    print(df.describe())
    
    # Summary for categorical (object) columns
    print("\n" + "=" * 50)
    print("CATEGORICAL SUMMARY")
    print("=" * 50)
    print(df.describe(include=['object']))
    
    # Show unique values for columns with low number of distinct values
    # Useful for detecting categories, errors, or inconsistencies
    print("\n" + "=" * 50)
    print("UNIQUE VALUES (LOW CARDINALITY)")
    print("=" * 50)
    for col in df.columns:
        if df[col].nunique() < 10:  # Only show columns with few unique values
            print(f"{col}: {df[col].unique()}")
            
    # correlation
    print("\n" + "=" * 50)
    print("CORRELATION MATRIX")
    print("=" * 50)
    print(df.corr(numeric_only=True))
    
    # Detect outliers using the IQR (Interquartile Range) method
    print("\n" + "=" * 50)
    print("OUTLIERS CHECK (IQR METHOD)")
    print("=" * 50)
    
    # Loop through only numerical columns
    for col in df.select_dtypes(include=np.number).columns:
        Q1 = df[col].quantile(0.25)  # 25th percentile
        Q3 = df[col].quantile(0.75)  # 75th percentile
        IQR = Q3 - Q1  # Interquartile range
        
        # Count rows that fall outside the normal range
        outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
        print(f"{col}: {len(outliers)} outliers")

# **Load dataset**
Apply Data Cleaning Check Template

In [3]:
dataset = pd.read_csv("/kaggle/input/datasets/d3lhomi10/medical-cost-personal-dataset/insurance.csv")
data_quality_report(dataset)

DATA QUALITY REPORT
BASIC INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB
None

SHAPE OF DATA
Rows: 1338, Columns: 7

MISSING VALUES
Series([], dtype: int64)

DUPLICATES
Duplicate rows: 1

DATA TYPES
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

NUMERICAL SUMMARY
               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397 

# **Cleaning the data** 

### remove the duplication

In [4]:
# Step 2: Remove duplicates FROM the cleaned dataset
dataset_clean = dataset.drop_duplicates().copy()

# Final result
print(dataset_clean)

      age     sex     bmi  children smoker     region      charges
0      19  female  27.900         0    yes  southwest  16884.92400
1      18    male  33.770         1     no  southeast   1725.55230
2      28    male  33.000         3     no  southeast   4449.46200
3      33    male  22.705         0     no  northwest  21984.47061
4      32    male  28.880         0     no  northwest   3866.85520
...   ...     ...     ...       ...    ...        ...          ...
1333   50    male  30.970         3     no  northwest  10600.54830
1334   18  female  31.920         0     no  northeast   2205.98080
1335   18  female  36.850         0     no  southeast   1629.83350
1336   21  female  25.800         0     no  southwest   2007.94500
1337   61  female  29.070         0    yes  northwest  29141.36030

[1337 rows x 7 columns]


# **Select Independent Variable (X) and Dependent Variable (y)**

Note: The concepts of Independent and Dependent variables were discussed earlier.

If you would like to review or gain a clearer understanding, please refer to this section: [Go to Independent variable VS Dependent variable Section](https://github.com/Hazem1695/ml-concept-briefs)

In [5]:
# Separate independent variable(features (X)) by selecting all rows and all columns except the last one
X = dataset_clean.iloc[:, :-1].values
# Separate the dependent variable (Target(y)) by selecting the last column
y = dataset_clean.iloc[:, -1].values

In [6]:
y_log = np.log(y)
print(y_log)

[ 9.73417643  7.45330245  8.40053847 ...  7.39623314  7.60486709
 10.27991376]


# **Encoding categorical data**

## Encoding the Indpendent Variable Using One Hot Encoding

Note: I explained One-Hot Encoding earlier.

If you missed it or want a deeper understanding of how it works, you can find it here: [Go to One-Hot Encoding Section](https://github.com/Hazem1695/ml-concept-briefs)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
# One-hot encode the 'color' column (index 2) while keeping other features
# Initialize ColumnTransformer to handle categorical data
# [2] targets the 'color' column (index 2)
ct = ColumnTransformer(transformers = [('encoder', OneHotEncoder(sparse_output=False),[1,4,5])], remainder = 'passthrough')
# Execute transformation and convert to NumPy ndarray for model training
X = np.array(ct.fit_transform(X))

# **Splitting the dataset into the Train set and Test set**

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train , y_test = train_test_split(X, y_log , test_size = 0.2, random_state = 0)

# **Random Forest Regression Model**

## Training the Building Random Forest Regression model on the Training set

In [9]:
from sklearn.ensemble import RandomForestRegressor

# n_estimators = 10: create 10 trees each tree is an estimator

regressor = RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_split=5, min_samples_leaf=3, random_state=0)
regressor.fit(X_train, y_train)

RandomForestRegressor(max_depth=5, min_samples_leaf=3, min_samples_split=5,
                      random_state=0)

# **Preticting the Test set results**

In [10]:
y_pred_log = regressor.predict(X_test).reshape(-1,1)

y_pred = np.exp(y_pred_log)
np.set_printoptions(precision=2,suppress=True)
print(np.concatenate((y_pred.reshape(len(y_pred), 1), np.exp(y_test).reshape(len(y_test), 1)), axis = 1))

[[ 2151.97  1633.96]
 [ 9943.32  8547.69]
 [10171.06  9290.14]
 [34768.72 32548.34]
 [10459.39  9644.25]
 [ 4045.28  2680.95]
 [ 2450.57  2198.19]
 [ 1873.9   1241.56]
 [ 2434.84  2710.83]
 [12227.59 12235.84]
 [ 9862.64  8280.62]
 [18435.85 17043.34]
 [13784.34 13974.46]
 [ 9040.7   8219.2 ]
 [ 5771.06  5472.45]
 [ 3993.76  2438.06]
 [ 5369.77  5267.82]
 [ 6089.16  3490.55]
 [ 7510.7   6640.54]
 [13920.04 14692.67]
 [ 2158.07  1622.19]
 [14066.78 13224.69]
 [ 1919.89  1256.3 ]
 [ 2629.65  2643.27]
 [ 1948.04  1674.63]
 [ 6963.67  4667.61]
 [ 4604.63  3732.63]
 [10341.63 11552.9 ]
 [ 3705.97  3756.62]
 [36985.04 37465.34]
 [ 7312.81  8059.68]
 [46338.84 47462.89]
 [12188.57 10577.09]
 [12091.59 20630.28]
 [16982.13 14571.89]
 [14730.39 36580.28]
 [ 9363.92  8347.16]
 [37654.94 51194.56]
 [10308.16  8428.07]
 [ 2172.91  1880.49]
 [34759.75 33475.82]
 [ 3535.26  2867.12]
 [ 6843.73  4564.19]
 [47969.47 47496.49]
 [37325.62 36149.48]
 [ 9831.19  8125.78]
 [10316.65 19749.38]
 [ 7312.81  7

# **Evaluating the Model Performance**

In [11]:
from sklearn.metrics import r2_score
r2_score(y_test, y_pred_log)

0.8561055893511798

In [12]:
y_trained = regressor.predict(X_train)
r2_score(y_train, y_trained)

0.8542448686522618

# **Model Comparison Table**

| Model Configuration           | Test R²    | Train R²   | Observation         |
| ----------------------------- | ---------- | ---------- | ------------------- |
| Default (no constraints)      | 0.8348     | 0.9760     | Severe overfitting  |
| max_depth=4                   | 0.8477     | 0.8776     | Good generalization |
| max_depth=5                   | 0.8518     | 0.8895     | Strong balance      |
| max_depth=6                   | 0.8501     | 0.9032     | Increasing overfit  |
| max_depth=7                   | 0.8483     | 0.9143     | More overfitting    |
| max_depth=5, n_estimators=200 | 0.8511     | 0.8897     | No real improvement |
| **max_depth=5 + log**         | **0.8561** | **0.8542** | **Best performance**|

---

# **Important Insights**

##### **1. Impact of Model Complexity**

Increasing tree depth improved performance up to a certain point (depth = 5), after which the model began to overfit. This is evident from the rising gap between training and testing scores at higher depths. The default model showed extreme overfitting, proving that uncontrolled complexity harms generalization.

##### **2. Bias–Variance Tradeoff**

The best non log model (max_depth=5) achieved a good balance between bias and variance, but still showed a noticeable gap between train and test performance. As depth increased beyond this point, variance dominated, leading to reduced generalization despite higher training scores.

##### **3. Effect of Increasing Number of Trees**

Doubling the number of estimators from 100 to 200 did not significantly improve performance. This indicates that the model had already reached stability, and further increasing ensemble size only added computational cost without meaningful gain.

##### **3. Effect of Log Transformation**

Applying log transformation to the target variable improved both performance and stability. It increased test R² to the highest value while simultaneously eliminating overfitting, as seen by nearly identical train and test scores. This suggests that transforming a skewed target distribution helps the model learn more generalizable patterns.

---
# **Conclusion**

The optimal model was achieved using a moderately constrained Random Forest (max_depth=5) combined with log transformation. This combination not only delivered the best predictive performance but also produced the most stable and generalizable model, highlighting that data transformation can be more impactful than increasing model complexity.

---

# **Model Comparison**

| Model Type                | Configuration                 | R² Score (Test) |
| ------------------------- | ----------------------------- | --------------- |
| Linear Regression         | Baseline                      | 0.7530          |
| **Linear Regression**     | **+ Log Transformation**      | **0.7716**      |
| Linear Regression         | + Log + Feature Scaling       | 0.7716          |
| Polynomial Regression     | Baseline                      | 0.8349          |
| **Polynomial Regression** | **+ Log Transformation**      | **0.8495**      |
| Polynomial Regression     | + Log + Feature Scaling       | 0.8495          |
| SVR                       | RBF + Feature Scaling         | 0.8377          |
| SVR                       | Polynomial (degree=2)         | 0.8347          |
| SVR                       | Polynomial (degree=3)         | 0.8235          |
| SVR                       | Polynomial (degree=4)         | 0.7935          |
| SVR                       | Linear Kernel                 | 0.7043          |
| SVR                       | Sigmoid Kernel                | -254.12         |
| **SVR**                   | **RBF + Log Transformation**  | **0.8556**      |
| Decision Tree             | No Tune, no log               | 0.6860          |
| Decision Tree             | Tuned, no log (max_depth=3)   | 0.8306          |
| Decision Tree             | Tuned, no log (max_depth=5)   | 0.8410          |
| Decision Tree             | Tuned, no log (max_depth=6)   | 0.8417          |
| Decision Tree             | Tuned, no log (max_depth=7)   | 0.8277          |
| **Decision Tree**         | **Tuned, log (max_depth=6)**  | **0.8539**      |
| Random Forest             | Default (no constraints)      | 0.8348          |
| Random Forest             | max_depth=4                   | 0.8477          |
| Random Forest             | max_depth=5                   | 0.8518          |
| Random Forest             | max_depth=6                   | 0.8501          |
| Random Forest             | max_depth=7                   | 0.8483          |
| Random Forest             | max_depth=5, n_estimators=200 | 0.8511          |
| **Random Forest**         | **max_depth=5 + log**         | **0.8561**      |
